# Notebook 07 — Hybrid molecular representations and cross-assay multi-task learning

**Question:** Do frozen CheMeleon graph embeddings, explicit RDKit descriptors, and auxiliary assay supervision provide *complementary* improvements in predicting direct CYP inhibition?

This notebook implements two controlled experiments on the **same five scaffold folds as Notebooks 05/06**:

**A. Hybrid representation:** LightGBM trained on [frozen CheMeleon embeddings + RDKit descriptors], optionally with Morgan count fingerprints. Its endpoint labels and held-out molecules are identical to those used in Notebook 05.

**B. Cross-assay learning:** three *frozen-encoder* neural heads, trained with masked multi-task losses: direct pIC50 only; auxiliary labels on the same primary molecules; and auxiliary labels **plus auxiliary-only molecules**. The last distinction separates task enrichment from extra molecular coverage. Direct-inhibition pIC50 is the only evaluation endpoint. Regression heads use SmoothL1/Huber; binary `is_TDI` heads use BCE-with-logits. **This is not end-to-end fine-tuning of CheMeleon.** Notebook 06 is the end-to-end fine-tuning comparator.

### Key safeguards
- Identify molecules by **canonical isomeric SMILES**. Never join assay labels solely on names.
- Exclude all training **assays** from outer test molecule/scaffold groups, including auxiliary-only molecules in the same held-out scaffold.
- Keep inner validation labels for early stopping separate from model fitting, including auxiliary data.
- Fit all preprocessing transforms and task normalizations on inner-training molecules only.
- Binary TDI labels are not treated as continuous inhibition measurements.
- The interval-distance diagnostic is **not** the official MA-ST-RAE.
- `FAST_MODE=True` is a one-fold functional check, **not** a result suitable for leaderboard comparisons.

Sources: [OpenADMET dataset](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test), [CheMeleon](https://github.com/JacksonBurns/chemeleon), [Chemprop](https://chemprop.readthedocs.io/).

## 0 · Install dependencies (Google Colab GPU recommended)

In [ ]:
%pip -q install "chemprop>=2.2.0,<2.4" "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.6" "scipy>=1.11" pandas matplotlib

## 1 · Settings and imports

In [ ]:
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import io, os, sys, json, hashlib, zipfile, time, random, platform, warnings, copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn as tnn
from torch.utils.data import DataLoader, TensorDataset
import lightgbm as lgb
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

SEED=2026
FAST_MODE=True                   # False: five-fold reproducible experiment
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(5))
N_SPLITS=5
RUN_HYBRID=True
RUN_CROSS_ASSAY=True
RUN_AUX_SAME_PRIMARY=True        # optional ablation: aux labels for primary structures only
RUN_AUX_ALL=True                 # primary + extra aux-only structures
MAX_EPOCHS=10 if FAST_MODE else 45
PATIENCE=3 if FAST_MODE else 9
BATCH_SIZE=128
LR=5e-4
HIDDEN=256
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']

# Optional CSV overrides for offline testing. Use the exact source filenames as keys.
DATA_OVERRIDES={}   # e.g. {'direct':'/content/direct.csv','single':'/content/screen.csv',...}
NOTEBOOK05_ZIP_PATH=None        # e.g. '/content/cyp_notebook_05_results.zip'
NOTEBOOK06_ZIP_PATH=None        # e.g. '/content/cyp_notebook_06_results.zip'
# Optional compatible embedding cache, generated by a previous Notebook 07 run.
# Optional: mount Google Drive first and set a persistent path to survive disconnects.
# from google.colab import drive; drive.mount('/content/drive')
PERSISTENT_ROOT=None  # e.g. '/content/drive/MyDrive/CYP_Notebook07'
BASE=Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
CACHE_DIR=BASE/'cyp_notebook_07_cache'
OUT=BASE/'cyp_notebook_07_results'
CACHE_DIR.mkdir(exist_ok=True);OUT.mkdir(exist_ok=True)
# Explicit synthetic-data / CI test switch. NEVER use True for scientific conclusions.
MOCK_EMBEDDINGS=False
# Optional override for ambiguous assay columns; see the printed column audit.
MANUAL_SCREEN_ISO_COL=None
MANUAL_SCREEN_VALUE_COL=None

ROOT_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'
FILES={
 'direct':'cyp-challenge-TRAIN_inhibition.csv',
 'single':'cyp-challenge-single-concentration-TRAIN.csv',
 'emax':'cyp-challenge-TRAIN_Emax.csv',
 'tdi':'cyp-challenge-TRAIN_TDI.csv'
}
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
print('Python',platform.python_version(),'Torch',torch.__version__,'RDKit',rdBase.rdkitVersion,'device',DEVICE)

## 2 · Retrieve and audit the original assay files

In [ ]:
def read_source(key):
    override=DATA_OVERRIDES.get(key)
    if override is None:
        with urlopen(ROOT_URL+FILES[key],timeout=150) as r: raw=r.read()
    else:
        raw=Path(override).read_bytes()
    return pd.read_csv(io.BytesIO(raw)),hashlib.sha256(raw).hexdigest()

sources={}; source_shas={}
for key in FILES:
    sources[key],source_shas[key]=read_source(key)
    print(f'\n{key}: {sources[key].shape}, SHA256={source_shas[key][:14]}')
    print('columns:',list(sources[key].columns))
RUN_SIGNATURE=hashlib.sha256(json.dumps({'sources':source_shas,'seed':SEED,
      'mock':MOCK_EMBEDDINGS,'epochs':MAX_EPOCHS,'lr':LR,'hidden':HIDDEN,
      'patience':PATIENCE,'batch':BATCH_SIZE},sort_keys=True).encode()).hexdigest()[:12]
print('Experiment signature:',RUN_SIGNATURE)
assert 'SMILES' in sources['direct'].columns
for iso in ISOFORMS:
    assert f'{iso}_pIC50_direct_inhibition' in sources['direct'].columns

## 3 · Canonical identities and fixed Notebook 05/06 folds

In [ ]:
def canon_scaf(s):
    if not isinstance(s,str) or not s.strip():return (None,None)
    mol=Chem.MolFromSmiles(s)
    if mol is None:return (None,None)
    canon=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(mol)
    grp=(Chem.MolToSmiles(core,canonical=True,isomericSmiles=False) if core.GetNumAtoms()
         else 'ACYCLIC_'+canon)
    return canon,grp

direct=sources['direct'].copy()
pairs=[canon_scaf(s) for s in direct.SMILES]
direct['canonical_smiles']=[a for a,b in pairs]
direct['scaffold_group']=[b for a,b in pairs]
direct=direct[direct.canonical_smiles.notna()].reset_index(drop=True)
assert direct.canonical_smiles.is_unique,'Resolve duplicate primary structures before CV.'
folds=np.full(len(direct),-1,dtype=int)
for k,(_,vi) in enumerate(GroupKFold(n_splits=N_SPLITS).split(direct,groups=direct.scaffold_group)):
    folds[vi]=k
direct['fold']=folds
assert (folds>=0).all()
manifest=direct[['canonical_smiles','scaffold_group','fold']].copy()
manifest.to_csv(OUT/'fold_manifest.csv',index=False)

# Optional independent verification against prior notebook exports.
def read_zip_file(zip_path,suffix):
    with zipfile.ZipFile(zip_path) as z:
        choices=[n for n in z.namelist() if n.endswith('/'+suffix) or n==suffix]
        if len(choices)!=1: raise ValueError(f'{zip_path}: expected one {suffix}, got {choices}')
        return z.read(choices[0])

for p in [NOTEBOOK05_ZIP_PATH,NOTEBOOK06_ZIP_PATH]:
    if p:
        earlier=pd.read_csv(io.BytesIO(read_zip_file(p,'fold_manifest.csv')))
        old=earlier.set_index('canonical_smiles')['fold']
        new=manifest.set_index('canonical_smiles')['fold']
        assert old.index.is_unique
        assert set(old.index)==set(new.index),'Molecule list mismatch with earlier notebook.'
        assert np.array_equal(old.loc[new.index].to_numpy(),new.to_numpy()),'Fold assignments differ.'
        meta=json.loads(read_zip_file(p,'metadata.json'))
        claimed=meta.get('dataset_sha256') or meta.get('data_sha256')
        if claimed:assert claimed==source_shas['direct'],'Dataset revision SHA mismatch.'
        print('Verified cross-notebook identities, folds and source hash for',p)
print('Primary structures',len(direct),'fold sizes',np.bincount(folds))

## 4 · Map auxiliary assays to compounds without assuming a fixed filename schema

In [ ]:
# Dataset schemas can evolve. Record every selected source column and its mapping.
# This cell intentionally rejects ambiguous selections instead of silently guessing.

primary_name_to_smiles={}
if 'Molecule_Name' in direct:
    primary_name_to_smiles=dict(zip(direct.Molecule_Name.astype(str),direct.canonical_smiles))

aux_sources={}
for source in ['single','emax','tdi']:
    frame=sources[source].copy()
    if 'SMILES' in frame:
        frame['canonical_smiles']=[canon_scaf(s)[0] for s in frame.SMILES]
    elif 'Molecule_Name' in frame:
        frame['canonical_smiles']=frame.Molecule_Name.astype(str).map(primary_name_to_smiles)
        print(source,': no SMILES column; auxiliary-only structures cannot be recovered from IDs.')
    else:
        print('WARNING:',source,'has neither SMILES nor Molecule_Name; ignoring.')
        continue
    frame=frame[frame.canonical_smiles.notna()].copy()
    aux_sources[source]=frame
    print(source,'mappable rows',len(frame),'unique structures',frame.canonical_smiles.nunique())

# Key => (kind, source, original column, isoform). `kind` = 'reg' or 'binary'.
AUDIT=[]
aux_tables={}

def add_long_table(source,identifier,measurement,values,kind='reg'):
    values=pd.to_numeric(values,errors='coerce')
    tab=pd.DataFrame({'canonical_smiles':identifier,'value':values})
    tab=tab.dropna(subset=['canonical_smiles','value'])
    if kind=='binary':
        tab=tab[tab.value.isin([0,1])]
        # Conflicting replicate annotations are excluded, not silently averaged.
        agg=tab.groupby('canonical_smiles').value.agg(['min','max'])
        conflict=agg['min']!=agg['max']
        if conflict.any():
            print('Excluded',int(conflict.sum()),'conflicting binary labels from',source,measurement)
        agg=agg.loc[~conflict]
        return agg['min'].astype(float)
    return tab.groupby('canonical_smiles').value.mean()

def numeric_col_ok(f,c):
    return pd.to_numeric(f[c],errors='coerce').notna().sum()>=4

for source in ['emax','tdi']:
    if source not in aux_sources: continue
    f=aux_sources[source]
    for iso in ISOFORMS:
        for c in f.columns:
            lc=str(c).lower()
            if iso.lower() not in lc:continue
            # Exclude bounds, assay-fit uncertainty estimates and reported direct pIC50.
            if any(t in lc for t in ['_conf_', 'ci_low','ci_high','_std','_error','_sem','_se']):continue
            if 'is_tdi' in lc or 'tdi_flag' in lc:
                if source=='tdi' and iso not in ['CYP2D6','CYP3A4']:continue
                code=f'{source}:{iso}:is_TDI'
                vals=f[c].replace({'True':1,'False':0,'true':1,'false':0,True:1,False:0})
                series=add_long_table(source,f.canonical_smiles,c,vals,'binary')
                if len(series):
                    aux_tables[code]=('binary',series)
                    AUDIT.append(dict(task=code,kind='binary',source=source,column=c,n=len(series)))
            elif source=='emax' and 'emax' in lc and numeric_col_ok(f,c):
                # Keep direct-condition and TDI-condition Emax as separate endpoints.
                code=f'emax:{iso}:{c}'
                aux_tables[code]=('reg',add_long_table(source,f.canonical_smiles,c,f[c]))
                AUDIT.append(dict(task=code,kind='reg',source=source,column=c,n=len(aux_tables[code][1])))
            elif source=='tdi' and 'pic50' in lc and 'tdi' in lc and 'direct' not in lc and numeric_col_ok(f,c):
                code=f'tdi:{iso}:{c}'
                aux_tables[code]=('reg',add_long_table(source,f.canonical_smiles,c,f[c]))
                AUDIT.append(dict(task=code,kind='reg',source=source,column=c,n=len(aux_tables[code][1])))

# Single concentration is often in long format: one molecule x CYP row, numerical log2 fold-change.
if 'single' in aux_sources:
    f=aux_sources['single']
    iso_col=MANUAL_SCREEN_ISO_COL
    if iso_col is None:
        choices=[]
        for c in f.columns:
            if c in ('canonical_smiles','SMILES','Molecule_Name'):continue
            vals=f[c].dropna().astype(str).str.upper().str.replace(' ','',regex=False)
            match=vals.isin(ISOFORMS)
            if len(vals) and match.mean()>0.75:choices.append(c)
        if len(choices)==1:iso_col=choices[0]
    if iso_col is not None:
        value_col=MANUAL_SCREEN_VALUE_COL
        if value_col is None:
            options=[c for c in f.columns if ('log2' in c.lower() or 'fold_change' in c.lower() or 'foldchange' in c.lower())
                     and not any(x in c.lower() for x in ['fdr','p_value','adj','error','std'])
                     and numeric_col_ok(f,c)]
            if len(options)==1:value_col=options[0]
            elif options: print('Ambiguous single-concentration numerical columns:',options)
        if value_col is not None:
            iso_values=f[iso_col].astype(str).str.upper().str.replace(' ','',regex=False)
            for iso in ISOFORMS:
                mask=iso_values==iso
                code=f'screen:{iso}:{value_col}'
                series=add_long_table('single',f.loc[mask,'canonical_smiles'],value_col,f.loc[mask,value_col])
                if len(series):
                    aux_tables[code]=('reg',series)
                    AUDIT.append(dict(task=code,kind='reg',source='single',column=value_col,n=len(series)))
        else: print('WARNING: screen value column not uniquely detected; set MANUAL_SCREEN_VALUE_COL.')
    else:
        print('No long-format isoform column detected in screen; trying wide format.')
        for iso in ISOFORMS:
            cs=[c for c in f.columns if iso.lower() in c.lower() and
                ('log2' in c.lower() or 'fold_change' in c.lower()) and numeric_col_ok(f,c)]
            if len(cs)>1: print('WARNING ambiguous wide screen columns',iso,cs)
            if len(cs)==1:
                c=cs[0];code=f'screen:{iso}:{c}'
                series=add_long_table('single',f.canonical_smiles,c,f[c]);aux_tables[code]=('reg',series)
                AUDIT.append(dict(task=code,kind='reg',source='single',column=c,n=len(series)))

# Deduplicate the same logical TDI classification if both Emax and TDI files contain it.
# Prefer TDI-file labels for CYP2D6 and CYP3A4; otherwise use Emax labels.
for iso in ISOFORMS:
    candidates=[key for key in aux_tables if key.endswith(f'{iso}:is_TDI')]
    if len(candidates)>1:
        preferred=f'tdi:{iso}:is_TDI' if f'tdi:{iso}:is_TDI' in candidates else candidates[0]
        for key in candidates:
            if key!=preferred:aux_tables.pop(key)

AUDIT=pd.DataFrame([a for a in AUDIT if a['task'] in aux_tables]);AUDIT.to_csv(OUT/'auxiliary_column_audit.csv',index=False)
print('Selected auxiliary tasks:',len(aux_tables));display(AUDIT)
if not any(k.startswith('screen:') for k in aux_tables):
    warnings.warn('Screening data not mapped. Inspect printed columns; set manual column overrides.')
if not aux_tables: raise ValueError('No auxiliary assays mapped. Review schema audit before continuing.')

## 5 · Union of molecules, missing-label matrices and coverage audit

In [ ]:
all_canon=list(direct.canonical_smiles)
extra=sorted(set().union(*(set(s.index) for _,s in aux_tables.values())) - set(all_canon))
all_canon+=extra
all_index={s:i for i,s in enumerate(all_canon)}
N=len(all_canon); N_PRIMARY=len(direct)
all_groups=[canon_scaf(s)[1] for s in all_canon]
assert all(x is not None for x in all_groups)
core_targets=[f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS]
reg_tasks=['direct:'+iso for iso in ISOFORMS]+sorted(t for t,(kind,v) in aux_tables.items() if kind=='reg')
bin_tasks=sorted(t for t,(kind,v) in aux_tables.items() if kind=='binary')
YR=np.full((N,len(reg_tasks)),np.nan,dtype=np.float32)
YB=np.full((N,len(bin_tasks)),np.nan,dtype=np.float32)
YR[:N_PRIMARY,:4]=direct[core_targets].apply(pd.to_numeric,errors='coerce').to_numpy(np.float32)
for j,task in enumerate(reg_tasks[4:],start=4):
    for key,val in aux_tables[task][1].items():YR[all_index[key],j]=val
for j,task in enumerate(bin_tasks):
    for key,val in aux_tables[task][1].items():YB[all_index[key],j]=val
assert not np.isfinite(YR[N_PRIMARY:,:4]).any()
assert len(set(all_canon))==N
coverage=pd.DataFrame({'task':reg_tasks+bin_tasks,'type':['reg']*len(reg_tasks)+['binary']*len(bin_tasks),
                       'n_labeled':list(np.isfinite(YR).sum(axis=0))+list(np.isfinite(YB).sum(axis=0))})
coverage.to_csv(OUT/'task_coverage.csv',index=False)
display(coverage)
print('Total unique molecular structures:',N,'primary',N_PRIMARY,'auxiliary-only',len(extra))
# An auxiliary-only compound is used ONLY if it is outside held-out scaffold groups.

## 6 · Compute and cache RDKit descriptors and frozen CheMeleon embeddings

In [ ]:
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

hash_molecules=hashlib.sha256('\n'.join(all_canon).encode()).hexdigest()[:16]
FEATURE_CACHE=CACHE_DIR/f'features_{hash_molecules}_{rdBase.rdkitVersion}.npz'
if FEATURE_CACHE.exists():
    z=np.load(FEATURE_CACHE)
    desc=z['rdkit'];count=z['morgan'];desc_names=list(z['descriptor_names'])
else:
    desc_names=[name for name,fn in Descriptors._descList]
    desc_fns=[fn for name,fn in Descriptors._descList]
    desc=np.full((N,len(desc_names)),np.nan,dtype=np.float32)
    count=np.zeros((N,2048),dtype=np.float32)
    fp_gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048,countSimulation=False)
    for i,s in enumerate(all_canon):
        m=Chem.MolFromSmiles(s)
        count[i,:]=np.asarray(fp_gen.GetCountFingerprintAsNumPy(m),dtype=np.float32)
        for j,fn in enumerate(desc_fns):
            try:desc[i,j]=float(fn(m))
            except Exception:pass
        if i%2000==0:print('Descriptor feature',i,'/',N)
    desc[~np.isfinite(desc)]=np.nan
    np.savez_compressed(FEATURE_CACHE,rdkit=desc,morgan=count,descriptor_names=np.array(desc_names))
print('RDKit shape',desc.shape,'Morgan',count.shape)

EMB_CACHE=CACHE_DIR/f'chemeleon_{hash_molecules}_{"MOCK" if MOCK_EMBEDDINGS else "PRETRAINED"}.npy'
if EMB_CACHE.exists():
    emb=np.load(EMB_CACHE)
    assert emb.shape[0]==N
    print('Loaded cached CheMeleon embeddings',emb.shape)
elif MOCK_EMBEDDINGS:
    # Synthetic smoke test ONLY; not a pretrained model or scientific experimental result.
    rng=np.random.default_rng(SEED)
    emb=count @ rng.normal(size=(count.shape[1],48)).astype(np.float32) / np.sqrt(count.shape[1])
    emb=emb.astype(np.float32)
    np.save(EMB_CACHE,emb)
else:
    import chemprop
    from chemprop import featurizers, models, nn, data
    from chemprop.data import BatchMolGraph
    CKPT_URL='https://zenodo.org/records/15460715/files/chemeleon_mp.pt'
    CKPT_PATH=CACHE_DIR/'chemeleon_mp.pt'
    if not CKPT_PATH.exists():
        print('Downloading trusted author-released CheMeleon message-passing weights...')
        urlretrieve(CKPT_URL,CKPT_PATH)
    checkpoint=torch.load(CKPT_PATH,map_location='cpu',weights_only=True)
    mp=nn.BondMessagePassing(**checkpoint['hyper_parameters'])
    mp.load_state_dict(checkpoint['state_dict'])
    encoder=models.MPNN(message_passing=mp,agg=nn.MeanAggregation(),
         predictor=nn.RegressionFFN(input_dim=mp.output_dim),batch_norm=False).to(DEVICE)
    encoder.eval();feat=featurizers.SimpleMoleculeMolGraphFeaturizer()
    result=[]
    for start in range(0,N,64):
        bm=BatchMolGraph([feat(Chem.MolFromSmiles(s)) for s in all_canon[start:start+64]])
        bm.to(device=encoder.device)
        with torch.inference_mode(): part=encoder.fingerprint(bm).detach().cpu().numpy()
        result.append(part.astype(np.float32))
        if start%1280==0:print('Embedded',min(start+64,N),'of',N)
    emb=np.vstack(result)
    np.save(EMB_CACHE,emb)
    del encoder,mp
    if torch.cuda.is_available():torch.cuda.empty_cache()
assert emb.shape[0]==N and np.isfinite(emb).all()
print('CheMeleon embeddings:',emb.shape)

## 7 · Shared evaluator, fold manifest and optional historical baselines

In [ ]:
def scores(y_true,y_pred,lo=None,hi=None):
    yt=np.asarray(y_true,float);yp=np.asarray(y_pred,float)
    good=np.isfinite(yt)&np.isfinite(yp)
    if not good.any():return dict(n=0,mae=np.nan,r2=np.nan,spearman=np.nan,mean_interval_excess=np.nan)
    yt=yt[good];yp=yp[good]
    sp=float(spearmanr(yt,yp).statistic) if len(yt)>2 else np.nan
    s=dict(n=len(yt),mae=mean_absolute_error(yt,yp),
           r2=r2_score(yt,yp) if len(yt)>1 else np.nan,spearman=sp)
    if lo is not None:
        lo=np.asarray(lo,float)[good];hi=np.asarray(hi,float)[good]
        err=np.maximum(lo-yp,0)+np.maximum(yp-hi,0)
        err[(~np.isfinite(lo))|(~np.isfinite(hi))|(lo>hi)]=np.nan
        s['mean_interval_excess']=float(np.mean(err[np.isfinite(err)])) if np.isfinite(err).any() else np.nan
    else:s['mean_interval_excess']=np.nan
    return s

saved=[];run_stats=[];split_audit=[]
PARTIALS=OUT/'partials';PARTIALS.mkdir(parents=True,exist_ok=True)
def partial_file(method,fold):
    return PARTIALS/f'{RUN_SIGNATURE}_{method}_fold{fold}.csv'
def already_done(method,fold):
    file=partial_file(method,fold)
    if not file.exists():return False
    old=pd.read_csv(file)
    needed={'model','fold','isoform','canonical_smiles','y_true','y_pred','ci_low','ci_high'}
    assert needed.issubset(old.columns)
    assert (old.model==method).all() and (old.fold==fold).all()
    assert old.canonical_smiles.isin(direct.canonical_smiles).all()
    assert not old.duplicated(['model','fold','isoform','canonical_smiles']).any()
    saved.extend(old.to_dict('records'))
    for iso,sub in old.groupby('isoform'):
        run_stats.append(dict(model=method,fold=fold,isoform=iso,
                   **scores(sub.y_true,sub.y_pred,sub.ci_low,sub.ci_high)))
    print('Resumed completed fold:',method,fold,'rows',len(old))
    return True
def save_outer_predictions(model,fold,test_idx,pred):
    assert pred.shape==(len(test_idx),4)
    for j,iso in enumerate(ISOFORMS):
        ids=test_idx[np.isfinite(YR[test_idx,j])]
        if not len(ids):continue
        ps=pred[np.isfinite(YR[test_idx,j]),j]
        lo=pd.to_numeric(direct.iloc[ids][f'{iso}_pIC50_direct_inhibition_conf_low'],errors='coerce').to_numpy(float)
        hi=pd.to_numeric(direct.iloc[ids][f'{iso}_pIC50_direct_inhibition_conf_high'],errors='coerce').to_numpy(float)
        run_stats.append(dict(model=model,fold=fold,isoform=iso,
                             **scores(YR[ids,j],ps,lo,hi)))
        for idx,val,low,high in zip(ids,ps,lo,hi):
            saved.append(dict(model=model,fold=fold,isoform=iso,canonical_smiles=all_canon[idx],
                              y_true=float(YR[idx,j]),y_pred=float(val),ci_low=float(low),ci_high=float(high)))
    pd.DataFrame([r for r in saved if r['model']==model and r['fold']==fold]).to_csv(partial_file(model,fold),index=False)
    print(model,'fold',fold,'done; cached for resume')

def safe_outer_split(fold):
    test_idx=np.flatnonzero(folds==fold)
    train_idx=np.flatnonzero(folds!=fold)
    test_groups=set(np.array(all_groups,dtype=object)[test_idx])
    # All auxiliary-only compounds with a held-out scaffold are excluded from training.
    union_train=np.array([i for i in range(N) if all_groups[i] not in test_groups],dtype=int)
    assert set(test_idx).isdisjoint(union_train)
    assert not test_groups.intersection({all_groups[i] for i in union_train})
    assert set(train_idx).issubset(union_train)
    return train_idx,test_idx,union_train

def historical_preds(path,notebook_num,expected_models=None):
    if not path:return pd.DataFrame()
    name='oof_predictions.csv' if notebook_num==5 else 'predictions.csv'
    h=pd.read_csv(io.BytesIO(read_zip_file(path,name)))
    expected={'canonical_smiles','fold','isoform','model','y_true','y_pred'}
    assert expected.issubset(h.columns)
    h=h[h['model'].isin(expected_models)] if expected_models else h
    h=h.copy();h['model']='Historical_'+h['model']
    return h

historical=[]
for path,n,expected in [(NOTEBOOK05_ZIP_PATH,5,['Morgan_count_RDKit','ECFP4_RDKit_chemistry']),
                      (NOTEBOOK06_ZIP_PATH,6,['CheMeleon_finetuned'])]:
    if path:historical.append(historical_preds(path,n,expected))
historical=pd.concat(historical,ignore_index=True) if historical else pd.DataFrame()

## 8 · Experiment A: frozen CheMeleon + descriptors + Morgan counts, LightGBM

In [ ]:
if RUN_HYBRID:
    Xhyb={
      'Hybrid_Emb_RDKit':np.concatenate([emb,desc],axis=1),
      'Hybrid_Emb_Morgan_RDKit':np.concatenate([emb,count,desc],axis=1)
    }
    for fold in FOLDS_TO_RUN:
        tr,te,union_train=safe_outer_split(fold)
        for method,X in Xhyb.items():
            if already_done(method,fold):continue
            pred=np.full((len(te),4),np.nan)
            for j,iso in enumerate(ISOFORMS):
                ids=tr[np.isfinite(YR[tr,j])]
                if len(ids)<15:continue
                model=lgb.LGBMRegressor(n_estimators=300,learning_rate=.04,num_leaves=15,
                        min_child_samples=15,reg_lambda=3,verbosity=-1,n_jobs=2,
                        random_state=SEED)
                model.fit(X[ids],YR[ids,j])
                pred[:,j]=model.predict(X[te])
            save_outer_predictions(method,fold,te,pred)

## 9 · Experiment B: frozen-encoder multi-assay network with mixed losses

**Exactly one architecture and fixed initial seed per fold:**

1. `MLP_direct_only`: four direct pIC50 tasks on original primary training structures.
2. `MLP_aux_same_primary`: same structures but additional experimental labels.
3. `MLP_aux_all`: all eligible primary + auxiliary-only structures (no validation scaffold exposure).

All share the same frozen CheMeleon embeddings and RDKit descriptors. Task-specific **masked Huber regression** is used for quantitative readouts and **BCE** for `is_TDI` labels. Direct inhibition has primary loss weight 1. The auxiliary-loss weights are exploratory and should be sensitivity-tested before drawing firm conclusions.

In [ ]:
class AssayNet(tnn.Module):
    def __init__(self,n_features,n_reg,n_binary):
        super().__init__()
        self.trunk=tnn.Sequential(tnn.Linear(n_features,HIDDEN),tnn.LayerNorm(HIDDEN),tnn.GELU(),
                                  tnn.Dropout(.15),tnn.Linear(HIDDEN,HIDDEN//2),tnn.GELU())
        self.reg=tnn.Linear(HIDDEN//2,n_reg)
        self.binary=tnn.Linear(HIDDEN//2,n_binary) if n_binary else None
    def forward(self,x):
        z=self.trunk(x)
        return self.reg(z), self.binary(z) if self.binary is not None else None

def taskwise_mean(loss,valid):
    counts=valid.sum(dim=0).clamp(min=1)
    each=(loss*valid).sum(dim=0)/counts
    keep=(valid.sum(dim=0)>0)
    return each[keep].mean() if bool(keep.any()) else loss.sum()*0.0

def mixed_loss(yhat_r,yhat_b,yr,yb,train_aux):
    rm=torch.isfinite(yr)
    # Scaled targets. In contrast, binary heads remain 0/1.
    safe_r=torch.nan_to_num(yr,nan=0.0)
    huber=torch.nn.functional.smooth_l1_loss(yhat_r,safe_r,reduction='none',beta=1.0)
    direct=taskwise_mean(huber[:,:4],rm[:,:4].float())
    if not train_aux:return direct
    extras=[]
    reg_groups={'screen':[],'emax':[],'tdi':[]}
    for j,t in enumerate(reg_tasks[4:],start=4):
        reg_groups[t.split(':')[0]].append(j)
    for key,weight in [('screen',.20),('emax',.10),('tdi',.10)]:
        cols=reg_groups[key]
        if cols:extras.append(weight*taskwise_mean(huber[:,cols],rm[:,cols].float()))
    if yhat_b is not None and yb.shape[1]:
        bm=torch.isfinite(yb)
        target=torch.nan_to_num(yb,nan=0.0)
        bce=torch.nn.functional.binary_cross_entropy_with_logits(yhat_b,target,reduction='none')
        extras.append(.10*taskwise_mean(bce,bm.float()))
    return direct+sum(extras) if extras else direct

def make_inner_split(train_primary,fold):
    splitter=GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED+fold)
    a,b=next(splitter.split(train_primary,groups=direct.scaffold_group.iloc[train_primary]))
    return train_primary[a],train_primary[b]

# Descriptor transform is fitted only on actual inner-training input structures.
# Input normalization also applies to embeddings (important for frozen graph features).
Xbase=np.concatenate([emb,desc],axis=1).astype(np.float32)

def fit_features(fit_idx):
    values=Xbase[fit_idx]
    med=np.nanmedian(values,axis=0)
    med[~np.isfinite(med)]=0
    x=np.where(np.isfinite(Xbase),Xbase,med[None,:])
    scaler=StandardScaler().fit(x[fit_idx])
    return np.clip(scaler.transform(x),-8,8).astype(np.float32)

def fit_target_norm(indices):
    means=np.zeros(len(reg_tasks),dtype=np.float32)
    stds=np.ones(len(reg_tasks),dtype=np.float32)
    for j in range(len(reg_tasks)):
        yy=YR[indices,j];yy=yy[np.isfinite(yy)]
        if len(yy)>1:
            means[j]=float(np.mean(yy))
            stds[j]=max(float(np.std(yy)),.05)
    return means,stds

def train_one(fold,kind):
    # Seed reset guarantees identical initial architecture for ablation arms on a fold.
    torch.manual_seed(SEED+fold);np.random.seed(SEED+fold)
    tr,te,eligible=safe_outer_split(fold)
    fit_primary,dev_idx=make_inner_split(tr,fold)
    dev_groups={all_groups[i] for i in dev_idx}
    if kind=='direct': fit_idx=fit_primary; train_aux=False
    elif kind=='same_primary': fit_idx=fit_primary;train_aux=True
    elif kind=='all':
        fit_idx=np.asarray([i for i in eligible if all_groups[i] not in dev_groups],dtype=int)
        train_aux=True
    else:raise ValueError(kind)
    assert set(fit_primary).issubset(fit_idx)
    assert not set(fit_idx).intersection(te)
    assert not {all_groups[i] for i in fit_idx}.intersection({all_groups[i] for i in np.r_[dev_idx,te]})
    split_audit.append(dict(fold=fold,model=kind,n_fit=len(fit_idx),n_inner_dev=len(dev_idx),
                            n_test=len(te),n_extra_fit=int((fit_idx>=N_PRIMARY).sum())))
    X=fit_features(fit_idx)
    means,stds=fit_target_norm(fit_idx)
    yn=(YR-means[None,:])/stds[None,:]
    if kind=='direct':
        # Mask ALL auxiliary labels in this arm without removing their output dimensions.
        yn=yn.copy();yn[:,4:]=np.nan
        yb=np.full_like(YB,np.nan)
    else:yb=YB
    dataset=TensorDataset(torch.from_numpy(X[fit_idx]),torch.from_numpy(yn[fit_idx]),
                          torch.from_numpy(yb[fit_idx]))
    loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    model=AssayNet(X.shape[1],len(reg_tasks),len(bin_tasks)).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-3)
    X_dev=torch.from_numpy(X[dev_idx]).to(DEVICE)
    yt_dev=YR[dev_idx,:4];mask_dev=np.isfinite(yt_dev)
    best_metric=np.inf;best_state=None;bad=0;history=[]
    for epoch in range(MAX_EPOCHS):
        model.train();losses=[]
        for xb,yr,bb in loader:
            xb=xb.to(DEVICE);yr=yr.to(DEVICE);bb=bb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pr,pb=model(xb)
            loss=mixed_loss(pr,pb,yr,bb,train_aux)
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite training loss')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step();losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad():pred_dev=model(X_dev)[0].cpu().numpy()[:,:4]*stds[:4]+means[:4]
        per=[float(np.mean(np.abs(yt_dev[mask_dev[:,j],j]-pred_dev[mask_dev[:,j],j])))
             for j in range(4) if mask_dev[:,j].any()]
        dev_score=float(np.mean(per)) if per else np.inf
        history.append(dict(model=kind,fold=fold,epoch=epoch,fit_loss=float(np.mean(losses)),dev_macro_mae=dev_score))
        if np.isfinite(dev_score) and dev_score<best_metric-1e-4:
            best_metric=dev_score
            best_state={key:val.detach().cpu().clone() for key,val in model.state_dict().items()}
            bad=0
        else:bad+=1
        if bad>=PATIENCE:break
    if best_state is None:raise RuntimeError('No finite inner validation score.')
    model.load_state_dict(best_state);model.to(DEVICE).eval()
    with torch.no_grad():
        p=model(torch.from_numpy(X[te]).to(DEVICE))[0].cpu().numpy()[:,:4]
    p=p*stds[:4]+means[:4]
    model_name={'direct':'MLP_direct_only','same_primary':'MLP_aux_same_primary','all':'MLP_aux_all'}[kind]
    checkpoint=OUT/f'{model_name}_fold{fold}.pt'
    torch.save({'state_dict':best_state,'settings':{'hidden':HIDDEN,'n_reg':len(reg_tasks),'n_bin':len(bin_tasks)},
                'mean':means,'std':stds,'fold':fold,'tasks':reg_tasks,'bin_tasks':bin_tasks},checkpoint)
    save_outer_predictions(model_name,fold,te,p)
    print('Fit',model_name,'epoch best inner MAE',round(best_metric,4),'n fit',len(fit_idx))
    return history

histories=[]
if RUN_CROSS_ASSAY:
    for fold in FOLDS_TO_RUN:
        if not already_done('MLP_direct_only',fold):histories+=train_one(fold,'direct')
        if RUN_AUX_SAME_PRIMARY and not already_done('MLP_aux_same_primary',fold):
            histories+=train_one(fold,'same_primary')
        if RUN_AUX_ALL and not already_done('MLP_aux_all',fold):
            histories+=train_one(fold,'all')
pd.DataFrame(histories).to_csv(OUT/'training_history.csv',index=False)
pd.DataFrame(split_audit).to_csv(OUT/'leakage_and_split_audit.csv',index=False)

## 10 · Summaries, fixed 50:50 historical blend and residual comparison

In [ ]:
results=pd.DataFrame(saved)
assert len(results)>0,'No models enabled or no predictions produced.'
assert not results.duplicated(['model','fold','isoform','canonical_smiles']).any()
results.to_csv(OUT/'predictions.csv',index=False)
fold_metrics=pd.DataFrame(run_stats)
fold_metrics.to_csv(OUT/'fold_metrics.csv',index=False)
summary=[]
for (model,iso),part in results.groupby(['model','isoform']):
    summary.append(dict(model=model,isoform=iso,n_folds=part.fold.nunique(),
        **scores(part.y_true.to_numpy(),part.y_pred.to_numpy(),part.ci_low.to_numpy(),part.ci_high.to_numpy())))
summary=pd.DataFrame(summary);summary.to_csv(OUT/'summary.csv',index=False)
display(summary.round(4))

if not historical.empty:
    # Match *only* the exact same held-out molecule, isoform and fold.
    h=historical.copy()
    assert not h.duplicated(['model','fold','isoform','canonical_smiles']).any()
    ref=results[['model','fold','isoform','canonical_smiles','y_true','y_pred']].rename(
         columns={'model':'current_model','y_pred':'current_pred'})
    matched=ref.merge(h[['model','fold','isoform','canonical_smiles','y_true','y_pred']],
        on=['fold','isoform','canonical_smiles'],suffixes=('_current','_historical'))
    assert np.allclose(matched.y_true_current,matched.y_true_historical,atol=1e-5), 'Historical target mismatch!'
    matched.to_csv(OUT/'matched_historical_predictions.csv',index=False)
    print('Verified matched rows with earlier notebooks:',len(matched))
    # Fixed 50:50 blend is only computed for genuine previous Notebook 05 + 06 OOF results.
    lgb=h[h.model=='Historical_Morgan_count_RDKit']
    che=h[h.model=='Historical_CheMeleon_finetuned']
    if len(lgb) and len(che):
        join=lgb.merge(che,on=['fold','isoform','canonical_smiles'],suffixes=('_lgb','_che'))
        assert np.allclose(join.y_true_lgb,join.y_true_che,atol=1e-5)
        join['blend_pred']=.5*(join.y_pred_lgb+join.y_pred_che)
        join.to_csv(OUT/'fixed_50_50_blend.csv',index=False)
        print('Fixed blend, matched observations:',len(join))

fig,ax=plt.subplots(figsize=(12,5))
for model,p in summary.groupby('model'):
    ax.plot(p.isoform,p.mae,marker='o',label=model)
ax.set(ylabel='Scaffold-held-out MAE (pIC50)',title='Notebook 07: Hybrid features and auxiliary supervision')
ax.grid(alpha=.2);ax.legend(fontsize=8,ncol=2)
fig.tight_layout();fig.savefig(OUT/'01_model_comparison.png',dpi=160);plt.show()

# Difference between MLP ablations: positive means auxiliary model has lower MAE.
subset=results[results.model.isin(['MLP_direct_only','MLP_aux_same_primary','MLP_aux_all'])]
if not subset.empty:
    pivot=subset.pivot_table(index=['fold','isoform','canonical_smiles'],columns='model',values='y_pred')
    truth=subset.drop_duplicates(['fold','isoform','canonical_smiles']).set_index(['fold','isoform','canonical_smiles'])['y_true']
    rows=[]
    for model in ['MLP_aux_same_primary','MLP_aux_all']:
        if model not in pivot or 'MLP_direct_only' not in pivot:continue
        both=pivot[[model,'MLP_direct_only']].dropna()
        yt=truth.loc[both.index].to_numpy()
        rows.append(dict(comparison=model,n=len(both),
           direct_mae=float(mean_absolute_error(yt,both['MLP_direct_only'])),
           auxiliary_mae=float(mean_absolute_error(yt,both[model])),
           mae_improvement=float(np.mean(np.abs(yt-both['MLP_direct_only']))-np.mean(np.abs(yt-both[model])))))
    ablation=pd.DataFrame(rows);ablation.to_csv(OUT/'auxiliary_ablation.csv',index=False)
    display(ablation.round(4))

## 11 · Reproducibility metadata, cached artifacts, and downloadable result ZIP

In [ ]:
meta={
  'notebook':'07_hybrid_cross_assay_learning.ipynb',
  'dataset_sha256':source_shas,'run_signature':RUN_SIGNATURE, 'n_primary':N_PRIMARY,'n_union':N,
  'folds_run':FOLDS_TO_RUN,'n_splits':N_SPLITS,'complete_oof':sorted(FOLDS_TO_RUN)==list(range(N_SPLITS)),
  'fast_mode':FAST_MODE,'mock_embeddings':MOCK_EMBEDDINGS,
  'embedding_dim':int(emb.shape[1]),'rdkit_dim':int(desc.shape[1]),
  'tasks_reg':reg_tasks,'tasks_binary':bin_tasks,
  'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
  'python':platform.python_version(),'torch':torch.__version__,'rdkit':rdBase.rdkitVersion,
  'notes':['Frozen CheMeleon encoder; learned MLP heads; not end-to-end CheMeleon fine-tuning.',
      'No auxiliary labels from held-out molecules or scaffolds are used in training.',
      'Mean interval excess is not official MA-ST-RAE.',
      'A single outer fold in fast mode is an execution check, not a 5-fold benchmark.',
      'TDI bin labels use BCE-with-logits, not regression.']
}
(OUT/'metadata.json').write_text(json.dumps(meta,indent=2,default=str))
archive=Path('cyp_notebook_07_results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in OUT.iterdir():
        if file.is_file() and file.suffix not in ('.pt','.npy','.npz'):
            z.write(file,f'{OUT.name}/{file.name}')
print('Result archive:',archive.resolve(),f'{archive.stat().st_size/1024:.1f} KiB')
print('Frozen embedding cache:',EMB_CACHE.resolve())
print('Resume: set PERSISTENT_ROOT to mounted Google Drive, then rerun; finished fold CSVs are skipped.')

## 12 · Interpretation checklist

- **Representation:** Does `Hybrid_Emb_RDKit` improve over Notebook 05 `Morgan_count_RDKit` on matched folds and molecules? Does adding Morgan to embeddings help or only increase dimensionality?
- **Auxiliary tasks:** Is `MLP_aux_same_primary` better than `MLP_direct_only`? That isolates additional assay labels without adding molecular structures.
- **Extra coverage:** Is `MLP_aux_all` better than `MLP_aux_same_primary`? That tests the incremental value of auxiliary-only structures.
- **TDI:** We train on binary labels with a classification loss; this is auxiliary supervision, **not** a TDI classification score.
- **Leakage:** Review `leakage_and_split_audit.csv`, the auxiliary column audit, and the exact fold manifest. Held-out scaffold groups are never trained on.
- **Publication discipline:** Use `FAST_MODE=False` for full five-fold results; repeat with another outer split or independent holdout before claiming a robust improvement.
- **Future extension:** End-to-end CheMeleon fine-tuning with a mixed-regression/classification head would be a distinct experiment, not demonstrated by this frozen-encoder notebook.